# `treasury_curve(maturity, start, end=None, method="flat_forward")`

The US Treasury par yield and zero rate at any maturity from 1M to 30Y, per day, built from the tenors stored in `macro.treasury_rates` (FMP). Change the arguments and re-run any cell.

Each day's curve uses only that day's stored tenors: a missing tenor is never filled, and a maturity past the day's longest tenor gives no row that day. Yields of 1Y and shorter are taken as semiannual zero rates, longer ones as par yields of semiannual-coupon bonds (this module's assumption, not Treasury's published convention). Rates are in percent, compounded semiannually.

> **About the data shown.** This function's data comes from FMP's API, and FMP's license doesn't allow displaying it publicly. So the outputs saved here are synthetic: made-up data from [`_synthetic.py`](_synthetic.py), run through the function's own, unchanged code. The values are not real prices or yields.
>
> To see real data, start Jupyter with `ARORA_REAL_DATA=1`: every cell then reads prod, read-only, as `readonly_user`. Don't commit those outputs; `tests/test_notebooks_offline.py` fails if you save them.

In [1]:
# Synthetic data in place of FMP's (see the note above), unless Jupyter was started with ARORA_REAL_DATA=1.
import _synthetic

_synthetic.use()

Data: synthetic


In [2]:
import warnings
from datetime import date, timedelta

import polars as pl
from IPython.display import Markdown

from arora_data.dates import today_et
from arora_data.frames import to_df
from arora_data.sync import treasury, treasury_curve
from arora_data.treasury_curve import METHODS, CurvePoint


def show(rows):
    """One result or a list of results as a Markdown table."""
    rows = [r.model_dump() for r in (rows if isinstance(rows, list) else [rows])]
    if not rows:
        return Markdown("0 rows")
    head = list(rows[0])
    lines = ["|".join(head), "|".join("---" for _ in head)]
    lines += ["|".join(str(r[c]).replace("|", "/") for c in head) for r in rows]
    return Markdown(f"{len(rows)} rows\n\n" + "\n".join(lines))

## The methods
`flat_forward`, the default, bootstraps discount factors tenor by tenor and is linear in log discount factor between them, so the forward rate is flat between tenors. `monotone_convex` is Hagan and West's method, Treasury's own since 2021-12-06 (basic form, no positivity amendment): forwards are continuous and stay monotone where the tenors' average forwards are, and each day's tenors are solved together. `linear_par` draws straight lines between par yields and bootstraps zero rates from them; it warns, because its forward rates kink at each tenor. `cubic_par` runs a natural cubic spline through the par yields instead, then bootstraps the same way; it warns, because it can overshoot between tenors and every tenor moves the whole curve. `nss` fits a Nelson-Siegel-Svensson curve to each day's zero rates; it warns, because a fit misses the stored tenors (the 10Y by 2 bp on a typical day, up to 27 bp), and it needs 6 tenors that day.

In [3]:
pl.DataFrame({"method": list(METHODS), "warning": [w or "" for _, w in METHODS.values()]})

method,warning
str,str
"""flat_forward""",""""""
"""monotone_convex""",""""""
"""linear_par""","""linear_par is not arbitrage-fr…"
"""nss""","""nss is a fitted curve: it miss…"
"""cubic_par""","""cubic_par can overshoot betwee…"


## One maturity
`maturity="4Y"` over the last week. Maturities are months or years in any case and spacing: `"4Y"`, `"4y"`, `"48M"`, `"2.5Y"`. `lower` and `upper` are the stored tenors either side; at a stored tenor both are that tenor, and its `par_yield` is the stored yield.

In [4]:
week = today_et() - timedelta(days=7)
assert treasury_curve("4Y", week) == treasury_curve(" 4 y ", week)
show(treasury_curve("4Y", week))

6 rows

maturity|date|par_yield|zero_rate|lower|upper
---|---|---|---|---|---
4Y|2026-11-16|4.76515078709|4.78397194581|3Y|5Y
4Y|2026-11-17|4.73888745670|4.75689000889|3Y|5Y
4Y|2026-11-18|4.73262418709|4.75013421225|3Y|5Y
4Y|2026-11-19|4.71888701803|4.73678013429|3Y|5Y
4Y|2026-11-20|4.76262572437|4.78009751296|3Y|5Y
4Y|2026-11-23|4.76888849524|4.78697772428|3Y|5Y

## One day's curve
The latest day at a spread of maturities. Stored tenors come back as stored; par and zero part ways at the long end, where coupons are paid at lower rates than the final one.

In [5]:
day = treasury(today_et() - timedelta(days=7), tenor="10Y")[-1].date
maturities = ["1M", "3M", "6M", "9M", "1Y", "18M", "2Y", "3Y", "4Y", "5Y", "7Y", "10Y", "15Y", "20Y", "25Y", "30Y"]
points = [treasury_curve(m, day, day)[0] for m in maturities]
stored = {r.tenor: r.yield_pct for r in treasury(day, day)}
curve = to_df(points, CurvePoint).with_columns(stored=pl.Series([stored.get(m) for m in maturities]))
curve.select("maturity", "par_yield", "zero_rate", "stored", "lower", "upper")

maturity,par_yield,zero_rate,stored,lower,upper
str,"decimal[38,11]","decimal[38,11]","decimal[38,2]",str,str
"""1M""",3.96000000000,3.96000000000,3.96,"""1M""","""1M"""
"""3M""",4.02000000000,4.02000000000,4.02,"""3M""","""3M"""
"""6M""",4.11000000000,4.11000000000,4.11,"""6M""","""6M"""
"""9M""",4.20998775570,4.20998775570,null,"""6M""","""1Y"""
"""1Y""",4.26000000000,4.26000000000,4.26,"""1Y""","""1Y"""
…,…,…,…,…,…
"""10Y""",5.01000000000,5.04635091428,5.01,"""10Y""","""10Y"""
"""15Y""",5.07068314069,5.11443165388,null,"""10Y""","""20Y"""
"""20Y""",5.10000000000,5.14848049993,5.10,"""20Y""","""20Y"""


## The methods against each other
The same day. At stored tenors the par yields agree; between them they part, most where tenors are far apart (10Y to 20Y, 20Y to 30Y). Gaps are each method's par yield (and zero rate) minus flat_forward's, in basis points. Calling `linear_par` or `cubic_par` prints its warning, a `MethodWarning`, on every call.

In [6]:
between = ["9M", "18M", "4Y", "15Y", "25Y"]
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    other = {m: [treasury_curve(x, day, day, m)[0] for x in between] for m in ("monotone_convex", "linear_par", "cubic_par", "nss")}
flat = [p for p in points if p.maturity in between]
pl.DataFrame({
    "maturity": between, "flat_forward par": [p.par_yield for p in flat],
    **{f"{m} gap (bp)": [round((b.par_yield - a.par_yield) * 100, 2) for a, b in zip(flat, rows)] for m, rows in other.items()},
    **{f"{m} zero gap (bp)": [round((b.zero_rate - a.zero_rate) * 100, 2) for a, b in zip(flat, rows)] for m, rows in other.items()},
})

maturity,flat_forward par,monotone_convex gap (bp),linear_par gap (bp),cubic_par gap (bp),nss gap (bp),monotone_convex zero gap (bp),linear_par zero gap (bp),cubic_par zero gap (bp),nss zero gap (bp)
str,"decimal[38,11]","decimal[38,2]","decimal[38,2]","decimal[38,2]","decimal[38,2]","decimal[38,2]","decimal[38,2]","decimal[38,2]","decimal[38,2]"
"""9M""",4.20998775570,-2.06,-2.50,-2.02,-2.41,-2.06,-2.50,-2.02,-2.41
"""18M""",4.41284034925,-2.78,-3.78,-2.84,-2.77,-2.85,-3.87,-2.90,-2.83
"""4Y""",4.76888849524,-0.41,-2.39,-0.65,-0.97,-0.39,-2.50,-0.66,-1.00
"""15Y""",5.07068314069,0.02,-1.57,0.70,0.18,0.06,-1.85,0.95,0.22
"""25Y""",5.11225349942,0.08,-0.23,-0.11,0.02,0.14,0.18,-0.36,-0.02


In [7]:
show(treasury_curve("4Y", day, day, "linear_par") + treasury_curve("4Y", day, day, "cubic_par") + treasury_curve("4Y", day, day, "nss"))

3 rows

maturity|date|par_yield|zero_rate|lower|upper
---|---|---|---|---|---
4Y|2026-11-23|4.74500|4.76195825439|3Y|5Y
4Y|2026-11-23|4.76234181501|4.78038112612|3Y|5Y
4Y|2026-11-23|4.75916254379|4.77694042493|3Y|5Y

## nss misses the stored tenors
A fit, not an interpolation: at each stored tenor on the same day, nss's par yield against the stored one.

In [8]:
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    fitted = [treasury_curve(t, day, day, "nss")[0] for t in stored]
pl.DataFrame({"tenor": list(stored), "stored": list(stored.values()), "nss par": [p.par_yield for p in fitted],
              "miss (bp)": [round((p.par_yield - y) * 100, 2) for p, y in zip(fitted, stored.values())]})

tenor,stored,nss par,miss (bp)
str,"decimal[38,2]","decimal[38,11]","decimal[38,2]"
"""1M""",3.96,3.96072705604,0.07
"""2M""",3.99,3.99108008962,0.11
"""3M""",4.02,4.02084062493,0.08
"""6M""",4.11,4.10634375034,-0.37
"""1Y""",4.26,4.25945211271,-0.05
…,…,…,…
"""5Y""",4.84,4.83606163471,-0.39
"""7Y""",4.93,4.93374121927,0.37
"""10Y""",5.01,5.01201326988,0.20


## Full history
The 4Y, which Treasury does not publish, every day since 1990.

In [9]:
history = to_df(treasury_curve("4Y", date(1990, 1, 1)), CurvePoint)
history.group_by(pl.col("date").dt.year().alias("year")).agg(
    days=pl.len(), par_min=pl.col("par_yield").min(), par_max=pl.col("par_yield").max(),
    zero_last=pl.col("zero_rate").last()).sort("year")

year,days,par_min,par_max,zero_last
i32,u32,"decimal[38,11]","decimal[38,11]","decimal[38,11]"
1990,260,4.11642380008,4.80513655598,4.13886052956
1991,261,3.99520866208,4.82514235708,4.77315686377
1992,262,4.48765378372,5.59143030401,5.06898885834
1993,261,4.62282734350,5.30405251660,4.88205291527
1994,260,4.74898311832,5.31406321603,4.88626198155
…,…,…,…,…
2022,260,3.94893974568,4.81523669024,4.36555721918
2023,260,3.97148058548,4.89262618600,4.16610217356
2024,262,3.77773343846,4.27522580279,3.92482775103


## Gaps
A maturity past the day's longest tenor is skipped, never extrapolated: with no 30Y stored from 2002-02-19 to 2006-02-08, the 25Y has only the window's two ends. With no 20Y before 1993-10-01, the 15Y comes from the 10Y and 30Y. Before 2001-07-31 the first tenor is 3M, so 1M holds the 3M forward rate flat (`lower` is None).

In [10]:
show(treasury_curve("25Y", date(2002, 2, 14), date(2006, 2, 10)))

4 rows

maturity|date|par_yield|zero_rate|lower|upper
---|---|---|---|---|---
25Y|2002-02-14|4.86222956064|4.90934470714|20Y|30Y
25Y|2002-02-15|4.86222955341|4.90936577169|20Y|30Y
25Y|2006-02-09|4.99066344448|5.07025332889|20Y|30Y
25Y|2006-02-10|5.03451842507|5.11204308675|20Y|30Y

In [11]:
show(treasury_curve("15Y", date(1992, 6, 1), date(1992, 6, 3)) + treasury_curve("1M", date(1995, 6, 1), date(1995, 6, 1)))

4 rows

maturity|date|par_yield|zero_rate|lower|upper
---|---|---|---|---|---
15Y|1992-06-01|4.94542432769|4.97252387136|10Y|30Y
15Y|1992-06-02|4.91541874122|4.94263536335|10Y|30Y
15Y|1992-06-03|4.94057282037|4.96941496987|10Y|30Y
1M|1995-06-01|3.86000000000|3.86000000000|None|3M

## Bad input
A maturity outside 1M to 30Y, one that is not months or years, or an unknown method raises `ValueError` before the database is read.

In [12]:
for args in [("31Y", day), ("2W", day), ("10", day), ("4Y", date(2026, 1, 2), date(2026, 1, 1)), ("4Y", day, day, "cubic")]:
    try:
        treasury_curve(*args)
    except ValueError as e:
        print(e)

'31Y' is outside 1M to 30Y
'2W' is not a maturity: use months or years, such as 18M, 4Y or 2.5Y
'10' is not a maturity: use months or years, such as 18M, 4Y or 2.5Y
start must be on or before end
'cubic' is not a method: use one of flat_forward, monotone_convex, linear_par, nss, cubic_par
